In [ ]:
import json

with open("ecomdata.json") as f:
  data = json.load(f)

print(data)

[{'order_id': 1, 'customer': 'Alice Brown', 'order_date': '2025-08-01', 'status': 'Delivered', 'payment_method': 'Credit Card', 'shipping_address': '123 Main St, Quezon City', 'items': [{'product': 'Headphones', 'category': 'Electronics', 'price': 1500, 'quantity': 1}, {'product': 'USB-C Cable', 'category': 'Accessories', 'price': 250, 'quantity': 2}], 'total_amount': 2000}, {'order_id': 2, 'customer': 'Bob White', 'order_date': '2025-08-03', 'status': 'Pending', 'payment_method': 'Cash on Delivery', 'shipping_address': '45 Mabini St, Davao City', 'items': [{'product': 'Smartphone', 'category': 'Electronics', 'price': 22000, 'quantity': 1}, {'product': 'Phone Case', 'category': 'Accessories', 'price': 500, 'quantity': 1}], 'total_amount': 22500}, {'order_id': 3, 'customer': 'Charlie Green', 'order_date': '2025-08-05', 'status': 'Delivered', 'payment_method': 'GCash', 'shipping_address': '78 Rizal Ave, Cebu City', 'items': [{'product': 'Book', 'category': 'Books', 'price': 450, 'quantit

In [ ]:
!pip install firebase-admin

In [ ]:
import firebase_admin
from firebase_admin import credentials, db

if not firebase_admin._apps:
    cred = credentials.Certificate("firebasekey.json")

    firebase_admin.initialize_app(cred, {
        "databaseURL": "https://e-com-f57a7-default-rtdb.asia-southeast1.firebasedatabase.app/"
    })

    print("Firebase connected successfully!")
else:
    print("Firebase already initialized.")

Firebase connected successfully!


In [ ]:
with open("ecomdata.json", "r") as file:
  data = json.load(file)
print("JSON file loaded")
ref = db.reference("orders")
for order in data:
    order_id = str(order["order_id"]).strip()
    if ref.child(order_id).get() is None:
        ref.child(order_id).set(order)
print("Data uploaded successfully!")

JSON file loaded
Data uploaded successfully!


In [ ]:
books_ref = db.reference("orders")
data = books_ref.get()
print(data)

[None, {'customer': 'Alice Brown', 'items': [{'category': 'Electronics', 'price': 1500, 'product': 'Headphones', 'quantity': 1}, {'category': 'Accessories', 'price': 250, 'product': 'USB-C Cable', 'quantity': 2}], 'order_date': '2025-08-01', 'order_id': 1, 'payment_method': 'Credit Card', 'shipping_address': '123 Main St, Quezon City', 'status': 'Delivered', 'total_amount': 2000}, {'customer': 'Bob White', 'items': [{'category': 'Electronics', 'price': 22000, 'product': 'Smartphone', 'quantity': 1}, {'category': 'Accessories', 'price': 500, 'product': 'Phone Case', 'quantity': 1}], 'order_date': '2025-08-03', 'order_id': 2, 'payment_method': 'Cash on Delivery', 'shipping_address': '45 Mabini St, Davao City', 'status': 'Pending', 'total_amount': 22500}, {'customer': 'Charlie Green', 'items': [{'category': 'Books', 'price': 450, 'product': 'Book', 'quantity': 1}, {'category': 'Stationery', 'price': 300, 'product': 'Notebook Set', 'quantity': 2}], 'order_date': '2025-08-05', 'order_id': 3

In [ ]:
from datetime import datetime

def get_int(prompt):
    while True:
        try:
            return int(input(prompt))
        except ValueError:
            print("Invalid input. Please enter a number.")

def get_float(prompt):
    while True:
        try:
            return float(input(prompt))
        except ValueError:
            print("Invalid number. Try again.")

def get_valid_input(prompt, valid_options):
    while True:
        value = input(prompt).strip()
        if value.lower() in valid_options:
            return value
        print(f"Invalid input. Choose from: {', '.join(valid_options)}")

def get_valid_date(prompt):
    while True:
        date_str = input(prompt).strip()
        try:
            datetime.strptime(date_str, "%Y-%m-%d")
            return date_str
        except ValueError:
            print("Invalid date format. Use YYYY-MM-DD.")

filtered_data = [o for o in data if o is not None]
data = filtered_data

def crorders(data):
    name = input("Enter customer name: \n")
    results = []
    for order in data:
        if order["customer"].lower() == name.lower():
            results.append(order)
    return results

def print_order(order):
    print(f"\nOrder ID: {order['order_id']}")
    print(f"Customer: {order['customer']}")
    print(f"Order Date: {order['order_date']}")
    print(f"Status: {order['status']}")
    print(f"Payment Method: {order['payment_method']}")
    print(f"Address: {order['shipping_address']}")
    print("Items:")

    for item in order["items"]:
        print(f"  - {item['product']} (₱{item['price']} × {item['quantity']})")

    print(f"Total Amount: ₱{order['total_amount']}")
    print("-" * 40)

def sales_summary(data):
    orders = len(data)
    print(f"Total number of orders: {orders}")
    total_sales = 0
    for order in data:
        total_sales += sum(item['price'] * item['quantity'] for item in order['items'])
    print(f"Total sales amount: ₱{total_sales}")
    avg_value = total_sales / orders if orders > 0 else 0
    print(f"Average order value: ₱{avg_value}")

    menu()

def category_summary(data):
    category = input("Enter category to summarize (Electronics, Accessories, Books, Stationery, Sportswear): \n").lower()

    total_sales = 0
    total_quantity = 0
    found = False

    for order in data:
        for item in order['items']:
            if item['category'].lower() == category:
                found = True
                total_sales += item['price'] * item['quantity']
                total_quantity += item['quantity']

    if not found:
        print("Category doesn't exist.")
        menu()

    print(f"\nCategory: {category.capitalize()}")
    print(f"Total Sales: ₱{total_sales}")
    print(f"Total Quantity Sold: {total_quantity}")

    menu()

def pending_orders(data):

    print("\nPending Orders:")

    for order in data:
        if order['status'].lower() != "delivered":
            print_order(order)

    menu()

def best_product(data):
    sales = {}

    for order in data:
        for item in order['items']:
            product = item['product']
            qty = item['quantity']
            sales[product] = sales.get(product, 0) + qty

    sorted_products = sorted(sales.items(), key=lambda x: x[1], reverse=True)

    top_3 = sorted_products[:3]

    print("\nTop 3 Best-Selling Products:")
    for product, qty in top_3:
        print(f"• {product}: {qty} sold")

    menu()

def create_order():
    print("\n--- Create New Order ---")

    new_id = max(o['order_id'] for o in data) + 1 if data else 1

    customer = input("Customer name: ").strip()
    order_date = get_valid_date("Order date (YYYY-MM-DD): ")

    print("Status options: Pending, Shipped, Delivered")
    status = get_valid_input("Status: ", ["pending", "shipped", "delivered"])

    print("Payment options: Credit Card, Debit Card, Cash on Delivery, GCash")
    payment = get_valid_input("Payment method: ", ["credit card", "debit card", "cash on delivery", "gcash"])

    address = input("Shipping address: ").strip()

    items = []
    total = 0
    while True:
        print(f"\nItem #{len(items) + 1}")
        product = input("  Product name (or leave blank to stop): ").strip()
        if not product:
            if not items:
                print("  At least one item is required.")
                continue
            break
        category = input("  Category: ").strip()
        price = get_float("  Price (₱): ")
        quantity = get_int("  Quantity: ")

        items.append({
            "product": product,
            "category": category,
            "price": price,
            "quantity": quantity
        })
        total += price * quantity

    new_order = {
        "order_id": new_id,
        "customer": customer,
        "order_date": order_date,
        "status": status,
        "payment_method": payment,
        "shipping_address": address,
        "items": items,
        "total_amount": total
    }

    data.append(new_order)
    db.reference("orders").child(str(new_id)).set(new_order)
    print(f"\n✓ Order #{new_id} for {customer} created successfully.")
    edit_database_menu()

def edit_order():
    print("\n--- Edit Existing Order ---")
    order_id = get_int("Enter Order ID to edit: ")

    order = next((o for o in data if o['order_id'] == order_id), None)
    if not order:
        print(f"Order #{order_id} not found.")
        edit_database_menu()
        return

    print_order(order)
    print("What would you like to edit?")
    print("[1] Customer Name")
    print("[2] Order Date")
    print("[3] Status")
    print("[4] Payment Method")
    print("[5] Shipping Address")
    print("[6] Items")
    print("[7] Cancel")

    choice = get_int("Enter choice: ")

    if choice == 1:
        order['customer'] = input("New customer name: ").strip()
    elif choice == 2:
        order['order_date'] = get_valid_date("New order date (YYYY-MM-DD): ")
    elif choice == 3:
        print("Status options: Pending, Shipped, Delivered")
        order['status'] = get_valid_input("New status: ", ["pending", "shipped", "delivered"])
    elif choice == 4:
        print("Payment options: Credit Card, Debit Card, Cash on Delivery, GCash")
        order['payment_method'] = get_valid_input("New payment method: ", ["credit card", "debit card", "cash on delivery", "gcash"])
    elif choice == 5:
        order['shipping_address'] = input("New shipping address: ").strip()
    elif choice == 6:
        print("Current items:")
        for i, item in enumerate(order['items']):
            print(f"  [{i}] {item['product']} – ₱{item['price']} × {item['quantity']}")
        print("\nItem edit options:")
        print("[A] Add a new item")
        print("[E] Edit an existing item")
        print("[R] Remove an item")
        sub = input("Choice: ").strip().upper()

        if sub == 'A':
            product = input("  Product name: ").strip()
            category = input("  Category: ").strip()
            price = get_float("  Price (₱): ")
            quantity = get_int("  Quantity: ")
            order['items'].append({"product": product, "category": category,
                                   "price": price, "quantity": quantity})

        elif sub == 'E':
            idx = get_int("  Enter item index to edit: ")
            try:
                item = order['items'][idx]
            except IndexError:
                print("  Invalid index.")
                edit_database_menu()
                return
            item['product'] = input(f"  Product [{item['product']}]: ").strip() or item['product']
            item['category'] = input(f"  Category [{item['category']}]: ").strip() or item['category']
            new_price = input(f"  Price [₱{item['price']}]: ").strip()
            new_qty = input(f"  Quantity [{item['quantity']}]: ").strip()
            if new_price:
                try:
                    item['price'] = float(new_price)
                except ValueError:
                    print("  Invalid price, kept original.")
            if new_qty:
                try:
                    item['quantity'] = int(new_qty)
                except ValueError:
                    print("  Invalid quantity, kept original.")

        elif sub == 'R':
            idx = get_int("  Enter item index to remove: ")
            try:
                removed = order['items'].pop(idx)
                print(f"  Removed: {removed['product']}")
            except IndexError:
                print("  Invalid index.")
                edit_database_menu()
                return

    elif choice == 7:
        edit_database_menu()
        return
    else:
        print("Invalid choice.")
        edit_database_menu()
        return

    order['total_amount'] = sum(i['price'] * i['quantity'] for i in order['items'])
    db.reference("orders").child(str(order_id)).set(order)
    print(f"\n✓ Order #{order_id} updated successfully.")
    edit_database_menu()

def remove_order():
    print("\n--- Remove Order ---")
    order_id = get_int("Enter Order ID to remove: ")

    order = next((o for o in data if o['order_id'] == order_id), None)
    if not order:
        print(f"Order #{order_id} not found.")
        edit_database_menu()
        return

    print_order(order)
    confirm = input(f"Are you sure you want to delete Order #{order_id}? (yes/no): ").strip().lower()
    if confirm == 'yes':
        data.remove(order)
        db.reference("orders").child(str(order_id)).delete()
        print(f"✓ Order #{order_id} removed successfully.")
    else:
        print("Removal cancelled.")

    edit_database_menu()

def edit_database_menu():
    print("\n--- Edit Database ---")
    print("[1] Create Order")
    print("[2] Edit Order")
    print("[3] Remove Order")
    print("[4] Back to Main Menu")
    choice = get_int("Enter choice: \n")
    if choice == 1:
        create_order()
    elif choice == 2:
        edit_order()
    elif choice == 3:
        remove_order()
    elif choice == 4:
        menu()
    else:
        print("Invalid choice. Please try again.")
        edit_database_menu()

def menu():
    print("\nE-Shop Analyzer")
    print("Welcome! What would you like to do today?")
    print("[1] Sales Summary")
    print("[2] Top Products")
    print("[3] Category Summary")
    print("[4] View Pending Orders")
    print("[5] Search Orders by Customer Name")
    print("[6] Edit Database")
    print("[7] Exit Program")
    choice = get_int("Enter choice: \n")
    if choice == 1:
        sales_summary(data)
    elif choice == 2:
        best_product(data)
    elif choice == 3:
        category_summary(data)
    elif choice == 4:
        pending_orders(data)
    elif choice == 5:
        orders = crorders(data)
        for o in orders:
            print_order(o)
        menu()
    elif choice == 6:
        edit_database_menu()
    elif choice == 7:
        print("Exiting program. Goodbye!")
        exit()
    else:
        print("Invalid choice. Please try again.")
        menu()

menu()


E-Shop Analyzer
Welcome! What would you like to do today?
[1] Sales Summary
[2] Top Products
[3] Category Summary
[4] View Pending Orders
[5] Search Orders by Customer Name
[6] Edit Database
[7] Exit Program
Enter choice: 
1
Total number of orders: 4
Total sales amount: ₱29650
Average order value: ₱7412.5

E-Shop Analyzer
Welcome! What would you like to do today?
[1] Sales Summary
[2] Top Products
[3] Category Summary
[4] View Pending Orders
[5] Search Orders by Customer Name
[6] Edit Database
[7] Exit Program
Enter choice: 
2

Top 3 Best-Selling Products:
• USB-C Cable: 2 sold
• Notebook Set: 2 sold
• Headphones: 1 sold

E-Shop Analyzer
Welcome! What would you like to do today?
[1] Sales Summary
[2] Top Products
[3] Category Summary
[4] View Pending Orders
[5] Search Orders by Customer Name
[6] Edit Database
[7] Exit Program
Enter choice: 
3
Enter category to summarize (Electronics, Accessories, Books, Stationery, Sportswear): 
Electronics

Category: Electronics
Total Sales: ₱23500
T

KeyboardInterrupt: Interrupted by user